In [45]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

In [46]:
# Constants
Es = np.array([0,-np.log(2), 0])
kT =  1/3

In [47]:
Z = sum(np.exp(- Es / kT))

print(f"Z = {Z}")
Ps = np.zeros(3)
for i in range(3):
    Pi = 1 / Z*np.exp(Es[i] / kT)
    print(f"P{i} = {Pi}")

sum(Es*Ps)

Z = 10.000000000000002
P0 = 0.09999999999999998
P1 = 0.012499999999999994
P2 = 0.09999999999999998


np.float64(0.0)

In [48]:
rng = np.random.default_rng(0)

E = np.array([0, -np.log(2), 0])
kT = 1/3
steps = 1_000_000
state = 0
counts = np.zeros(3)

In [49]:
for _ in range(steps):
    proposed = rng.integers(3)
    dE = Es[proposed] - Es[state]
    if dE <= 0 or rng.random() < np.exp(-dE / kT):
        state = proposed
    counts[state] += 1

P_mc = counts / steps
E_mc = np.sum(P_mc * Es)
print(P_mc, E_mc)

[0.10029  0.799041 0.100669] -0.5538530163017993


In [50]:
T_matrix = np.array([
    np.array([0.5, 0.5,0]),
    np.array([0.5,0,0.5]),
    np.array([0,0.5,0.5])
])

In [51]:
state = 0
counts = np.zeros(3)

for _ in range(steps):
    trial = rng.choice(3, p=T_matrix[state])        # row i = proposal probs from state i
    dE = E[trial] - E[state]
    if dE <= 0 or rng.random() < np.exp(-dE / kT):
        state = trial
    counts[state] += 1

P_mc = counts / n_steps
print(P_mc, np.sum(P_mc * E))

[0.496745 4.001505 0.50175 ] -2.7736319087465238


In [52]:
class Node:
    def __init__(self, index, energy):
        self.index = index
        self.energy = energy

class System:
    def __init__(self, energies, kT, Tij):
        self.energies = energies
        self.kT = kT
        self.Tij = Tij
        self.nodes = [Node(i, e) for i, e in enumerate(self.energies)]
        self.state = self.nodes[0]

    def proposed_state(self):
        j = np.random.choice(len(self.nodes), p=self.Tij[self.state.index])  # use self.Tij[self.state.index] to get an index for the proposed state
        return self.nodes[j]

    def step(self):
        proposed_state = self.proposed_state()
        delta_e = proposed_state.energy - self.state.energy
        if delta_e <= 0 or np.random.rand() < np.exp(-delta_e / self.kT):
            self.state = proposed_state

    def run_Metropolis_Monte_Carlo(self, n_steps):
        for _ in range(n_steps):
            self.step()

In [54]:
Es = [0, -np.log(2), 0]
kT = 1/3
n_steps = 200_000

T_uniform = np.full((3, 3), 1/3)
T_varied = np.array([[0.5, 0.5, 0.0],
                     [0.5, 0.0, 0.5],
                     [0.0, 0.5, 0.5]])

# --- Exact results ---
beta = 1 / kT
boltzmann = np.exp(-beta * np.array(Es))
Q = boltzmann.sum()
P_exact = boltzmann / Q
E_exact = np.sum(P_exact * Es)
print(f"Exact: Q = {Q:.4f}, P = {P_exact}, <E> = {E_exact:.4f}")

# --- Monte Carlo for both proposal matrices ---
np.random.seed(0)
for name, T in [("Uniform", T_uniform), ("Varied", T_varied)]:
    system = System(Es, kT, T)
    counts = np.zeros(len(Es))
    for _ in range(n_steps):
        system.step()
        counts[system.state.index] += 1
    P_mc = counts / n_steps
    E_mc = np.sum(P_mc * Es)
    print(f"{name}: P = {P_mc}, <E> = {E_mc:.4f}")

# --- Optional: compare statistical noise over many seeds ---
n_runs = 20
n_short = 20_000
for name, T in [("Uniform", T_uniform), ("Varied", T_varied)]:
    p1 = []
    for seed in range(n_runs):
        np.random.seed(seed)
        s = System(Es, kT, T)
        visits_to_1 = 0
        for _ in range(n_short):
            s.step()
            if s.state.index == 1:
                visits_to_1 += 1
        p1.append(visits_to_1 / n_short)
    print(f"{name}: P1 mean = {np.mean(p1):.4f}, std = {np.std(p1):.4f}")

Exact: Q = 10.0000, P = [0.1 0.8 0.1], <E> = -0.5545
Uniform: P = [0.101425 0.797765 0.10081 ], <E> = -0.5530
Varied: P = [0.100065 0.800025 0.09991 ], <E> = -0.5545
Uniform: P1 mean = 0.7973, std = 0.0062
Varied: P1 mean = 0.7992, std = 0.0046
